<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/17-deep-reinforcement-learning-world-models.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Deep Reinforcement Learning and World Models** {#deep-reinforcement-learning-world-models}

Deep reinforcement learning (deep RL) studies agents whose data depends on their own decisions. A supervised learner receives a fixed collection of labeled pairs; an RL agent chooses an action, changes the state it will observe next, and may not receive useful feedback until much later. Neural networks provide scalable representations for policies, value functions, and environment models, but they also amplify bootstrapping error, distribution shift, and optimization instability.

![Actions change the future observations from which an RL agent learns.](assets/dl17-agent-environment-loop.svg){fig-align="center" width="76%" fig-alt="A closed loop shows an agent sending actions to an environment and receiving next observations, rewards, and termination signals."}

*Original mechanism diagram based on the agent-environment interface in Sutton and Barto's [Reinforcement Learning: An Introduction](https://mitpress.mit.edu/9780262039246/reinforcement-learning/).*

The executable thread uses the official [Gymnasium Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/) specification: observation $s_t=[\cos\theta_t,\sin\theta_t,\dot\theta_t]$, continuous torque $a_t\in[-2,2]$, 200-step episodes, and reward

$$
r_t=-\left(\theta_t^2+0.1\dot\theta_t^2+0.001a_t^2\right).
$$

The local implementation follows the documented equations so that every example runs offline without the optional Gymnasium dependency. One fixed set of trajectory seeds is split by whole episode into training, validation, and test partitions. Discrete-control methods use five torques from the same action interval; continuous-control methods retain the original action space. These short CPU experiments audit mechanisms rather than reproduce published benchmark scores.

![Gymnasium's coordinate system for Pendulum-v1.](assets/dl17-pendulum-coordinate.png){fig-align="center" width="46%" fig-alt="Official Pendulum-v1 coordinate diagram showing angle theta and applied torque."}

*Source: [Gymnasium Pendulum-v1 documentation](https://gymnasium.farama.org/environments/classic_control/pendulum/), Farama Foundation. The environment code and documentation source are distributed with the Gymnasium project under the project's MIT license.*

<details>
<summary><strong>PyTorch: establish the shared environment and episode-level data split</strong></summary>

```python
import copy
import math
import random
from collections import deque

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1717):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def angle_normalize(theta):
    return ((theta + math.pi) % (2 * math.pi)) - math.pi


class LocalPendulum:
    """Offline-compatible implementation of the documented Pendulum-v1 dynamics."""

    def __init__(self, horizon=120, g=10.0, mass=1.0, length=1.0, dt=0.05):
        self.horizon, self.g, self.mass, self.length, self.dt = horizon, g, mass, length, dt
        self.max_speed, self.max_torque = 8.0, 2.0

    def reset(self, seed):
        self.rng = np.random.default_rng(seed)
        self.theta = float(self.rng.uniform(-math.pi, math.pi))
        self.theta_dot = float(self.rng.uniform(-1.0, 1.0))
        self.steps = 0
        return self.observe()

    def observe(self):
        return np.array([math.cos(self.theta), math.sin(self.theta), self.theta_dot], dtype=np.float32)

    def step(self, action):
        torque = float(np.clip(np.asarray(action).reshape(-1)[0], -self.max_torque, self.max_torque))
        theta, theta_dot = self.theta, self.theta_dot
        cost = angle_normalize(theta) ** 2 + 0.1 * theta_dot ** 2 + 0.001 * torque ** 2
        acceleration = 3 * self.g / (2 * self.length) * math.sin(theta) + 3 * torque / (self.mass * self.length ** 2)
        self.theta_dot = float(np.clip(theta_dot + acceleration * self.dt, -8.0, 8.0))
        self.theta = float(theta + self.theta_dot * self.dt)
        self.steps += 1
        truncated = self.steps >= self.horizon
        return self.observe(), -float(cost), False, truncated


def scripted_action(observation, rng, random_probability=0.55):
    if rng.random() < random_probability:
        return float(rng.uniform(-2, 2))
    theta = math.atan2(float(observation[1]), float(observation[0]))
    return float(np.clip(-2.0 * theta - 0.45 * observation[2] + rng.normal(0, 0.25), -2, 2))


def collect_episode(seed, horizon=120):
    env, rng = LocalPendulum(horizon=horizon), np.random.default_rng(seed + 50_000)
    observation = env.reset(seed)
    records = []
    for _ in range(horizon):
        action = scripted_action(observation, rng)
        next_observation, reward, terminated, truncated = env.step(action)
        records.append((observation, action, reward, next_observation, terminated or truncated))
        observation = next_observation
    return records


seed_everything()
episode_seeds = np.arange(17_000, 17_090)
episodes = [collect_episode(int(seed)) for seed in episode_seeds]
train_episodes, val_episodes, test_episodes = episodes[:60], episodes[60:75], episodes[75:]


def flatten_episodes(selected):
    states = torch.tensor(np.stack([x[0] for ep in selected for x in ep]), dtype=torch.float32)
    actions = torch.tensor([[x[1]] for ep in selected for x in ep], dtype=torch.float32)
    rewards = torch.tensor([[x[2]] for ep in selected for x in ep], dtype=torch.float32)
    next_states = torch.tensor(np.stack([x[3] for ep in selected for x in ep]), dtype=torch.float32)
    dones = torch.tensor([[x[4]] for ep in selected for x in ep], dtype=torch.float32)
    return states, actions, rewards, next_states, dones


train_s, train_a, train_r, train_ns, train_done = flatten_episodes(train_episodes)
val_s, val_a, val_r, val_ns, val_done = flatten_episodes(val_episodes)
test_s, test_a, test_r, test_ns, test_done = flatten_episodes(test_episodes)

assert train_s.shape == (7200, 3) and train_a.shape == (7200, 1)
assert set(episode_seeds[:60]).isdisjoint(set(episode_seeds[75:]))
assert torch.allclose(train_s[:, :2].square().sum(1).mean(), torch.tensor(1.0), atol=1e-5)
print({"episode split": (60, 15, 15), "transition split": (len(train_s), len(val_s), len(test_s)), "reward range": (round(float(train_r.min()), 3), round(float(train_r.max()), 3))})
```

</details>

The split is made before any learned preprocessing or model fitting. Adjacent transitions within an episode are highly correlated, so randomly splitting individual rows would leak nearly identical states across partitions.


### **Sequential Decisions and Markov Decision Processes** {#sequential-decisions-mdps}

A Markov decision process (MDP) is a tuple

$$
\mathcal{M}=(\mathcal{S},\mathcal{A},P,R,\gamma,\rho_0).
$$

$\mathcal{S}$ and $\mathcal{A}$ are the state and action spaces; $P(s'|s,a)$ is the transition distribution; $R(s,a,s')$ specifies reward; $\gamma\in[0,1)$ discounts delayed rewards; and $\rho_0$ is the initial-state distribution. The Markov assumption says that the current state contains all information needed to predict the next state and reward:

$$
p(s_{t+1},r_t\mid s_0,a_0,\ldots,s_t,a_t)=p(s_{t+1},r_t\mid s_t,a_t).
$$

This is an assumption about the **state representation**, not necessarily the raw observation. A camera frame may hide velocity or objects outside the field of view, producing a partially observable MDP (POMDP). Frame stacking, recurrent state, belief-state inference, or a world model can restore useful memory.

At each step, the policy $\pi_\theta(a|s)$ produces an action distribution. The environment then generates $r_t,s_{t+1}$. Because $\pi_\theta$ influences which states are visited, an RL training set is endogenous: improving or destabilizing the policy changes the data distribution.

<details>
<summary><strong>Python: trace state, action, reward, termination, and truncation</strong></summary>

```python
trace_env = LocalPendulum(horizon=12)
observation = trace_env.reset(seed=1720)
trajectory = []
for step in range(12):
    action = scripted_action(observation, np.random.default_rng(1720 + step))
    next_observation, reward, terminated, truncated = trace_env.step(action)
    trajectory.append({"t": step, "state": observation.copy(), "action": action, "reward": reward, "terminated": terminated, "truncated": truncated})
    observation = next_observation

assert not any(item["terminated"] for item in trajectory)
assert trajectory[-1]["truncated"] and not trajectory[-2]["truncated"]
assert all(item["state"].shape == (3,) for item in trajectory)
print({"first transition": {"state": np.round(trajectory[0]["state"], 3).tolist(), "action": round(trajectory[0]["action"], 3), "reward": round(trajectory[0]["reward"], 3)}, "last step truncated": trajectory[-1]["truncated"]})
```

</details>

`terminated` means the underlying task reached a terminal state; `truncated` means an external limit stopped collection. Treating every time limit as true termination forces the value target to zero and can bias continuing-task estimates. A finite-horizon formulation avoids ambiguity by including remaining time in the state; otherwise the bootstrap convention must be stated explicitly.


### **Returns, Value Functions, and Bellman Equations** {#returns-values-bellman}

The discounted return from time $t$ is

$$
G_t=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.
$$

The state value $V^\pi(s)=\mathbb{E}_\pi[G_t|s_t=s]$ measures the expected return after following $\pi$. The action value $Q^\pi(s,a)$ conditions on one action before following the policy. Their difference, $A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s)$, is the advantage: whether the chosen action is better or worse than the policy's usual behavior at that state.

![A one-step Bellman target combines immediate reward with bootstrapped future value.](assets/dl17-bellman-backup.svg){fig-align="center" width="76%" fig-alt="The current estimate is compared with a target made from reward plus discounted value of the next state."}

The Bellman expectation equations are recursive consistency conditions:

$$
V^\pi(s)=\mathbb{E}_{a\sim\pi,s'\sim P}[r+\gamma V^\pi(s')],\qquad Q^\pi(s,a)=\mathbb{E}_{s'\sim P}[r+\gamma\mathbb{E}_{a'\sim\pi}Q^\pi(s',a')].
$$

Monte Carlo learning waits for $G_t$: low bootstrap bias but high variance and delayed updates. Temporal-difference (TD) learning uses $r_t+\gamma V(s_{t+1})$: earlier updates and lower variance, but bias from the current critic. Multi-step returns and generalized advantage estimation (GAE) interpolate between them.

<details>
<summary><strong>Python: compare return-to-go, one-step TD, and GAE</strong></summary>

```python
def discounted_returns(rewards, gamma=0.98):
    output = torch.zeros_like(rewards)
    running = torch.tensor(0.0)
    for index in range(len(rewards) - 1, -1, -1):
        running = rewards[index] + gamma * running
        output[index] = running
    return output


def generalized_advantages(rewards, values, next_values, gamma=0.98, lam=0.95):
    deltas = rewards + gamma * next_values - values
    advantages = torch.zeros_like(deltas)
    running = torch.tensor(0.0)
    for index in range(len(deltas) - 1, -1, -1):
        running = deltas[index] + gamma * lam * running
        advantages[index] = running
    return advantages


example_rewards = torch.tensor([item["reward"] for item in trajectory], dtype=torch.float32)
proxy_values = -2.0 * torch.arange(len(example_rewards), 0, -1, dtype=torch.float32)
proxy_next = torch.cat([proxy_values[1:], torch.zeros(1)])
returns = discounted_returns(example_rewards)
td_residuals = example_rewards + 0.98 * proxy_next - proxy_values
gae = generalized_advantages(example_rewards, proxy_values, proxy_next)
assert returns.shape == td_residuals.shape == gae.shape
assert torch.allclose(returns[-1], example_rewards[-1])
print({"G_0": round(float(returns[0]), 3), "mean absolute TD residual": round(float(td_residuals.abs().mean()), 3), "mean absolute GAE": round(float(gae.abs().mean()), 3)})
```

</details>

The proxy values above are deliberately imperfect, making the TD residual nonzero. In a real agent the critic is learned, so its error enters the actor's target. That coupling is one reason RL diagnostics must inspect returns, value loss, explained variance, entropy, and action distributions together.


### **Deep Q-Networks** {#deep-q-networks}

Q-learning seeks $Q^*(s,a)=\mathbb{E}[r+\gamma\max_{a'}Q^*(s',a')]$. A deep Q-network (DQN) represents $Q_\theta(s,a)$ with a neural network and minimizes a Huber or squared error against

$$
y=r+\gamma(1-d)\max_{a'}Q_{\bar\theta}(s',a').
$$

$d$ marks a terminal transition and $\bar\theta$ denotes a lagged target network. Three hazards meet here: function approximation generalizes updates across states, bootstrapping trains against another estimate, and off-policy replay contains actions from older policies. This combination can diverge.

![Replay and a lagged target network reduce two moving-target effects in DQN.](assets/dl17-dqn-stabilizers.svg){fig-align="center" width="78%" fig-alt="An online Q network, replay buffer, target Q network, and TD loss are connected in the DQN training pipeline."}

Experience replay reduces temporal correlation and reuses samples. A target network changes more slowly than the online network. Reward scaling, gradient clipping, Double DQN targets, dueling heads, and prioritized replay address additional failure modes. DQN naturally assumes a finite action set, so this example discretizes torque into five values.

<details>
<summary><strong>PyTorch: train a compact DQN on discretized Pendulum torque</strong></summary>

```python
class QNetwork(nn.Module):
    def __init__(self, action_count=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, action_count))

    def forward(self, state):
        return self.net(state)


seed_everything(1730)
discrete_torques = torch.linspace(-2, 2, 5)
dqn = QNetwork()
dqn_target = copy.deepcopy(dqn).eval()
dqn_optimizer = torch.optim.AdamW(dqn.parameters(), lr=1e-3)
replay = deque(maxlen=6000)
dqn_env = LocalPendulum(horizon=120)
state = dqn_env.reset(seed=1730)
rng = np.random.default_rng(1730)
dqn_losses = []

for global_step in range(5200):
    epsilon = max(0.08, 1.0 - global_step / 4200)
    if rng.random() < epsilon:
        action_index = int(rng.integers(5))
    else:
        with torch.no_grad():
            action_index = int(dqn(torch.tensor(state).unsqueeze(0)).argmax(1))
    next_state, reward, _, truncated = dqn_env.step(float(discrete_torques[action_index]))
    replay.append((state, action_index, reward / 10.0, next_state, truncated))
    state = next_state
    if truncated:
        state = dqn_env.reset(seed=1730 + global_step)

    if len(replay) >= 256:
        batch_ids = rng.integers(0, len(replay), size=64)
        batch = [replay[int(i)] for i in batch_ids]
        bs = torch.tensor(np.stack([x[0] for x in batch]), dtype=torch.float32)
        ba = torch.tensor([x[1] for x in batch], dtype=torch.long)
        br = torch.tensor([x[2] for x in batch], dtype=torch.float32)
        bns = torch.tensor(np.stack([x[3] for x in batch]), dtype=torch.float32)
        bd = torch.tensor([x[4] for x in batch], dtype=torch.float32)
        prediction = dqn(bs).gather(1, ba[:, None]).squeeze(1)
        with torch.no_grad():
            target = br + 0.98 * (1 - bd) * dqn_target(bns).max(1).values
        loss = F.smooth_l1_loss(prediction, target)
        dqn_optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(dqn.parameters(), 10.0)
        dqn_optimizer.step()
        dqn_losses.append(float(loss.detach()))
    if global_step % 160 == 0:
        dqn_target.load_state_dict(dqn.state_dict())


def dqn_policy(observation):
    with torch.no_grad():
        index = int(dqn(torch.tensor(observation, dtype=torch.float32).unsqueeze(0)).argmax(1))
    return float(discrete_torques[index])


assert len(replay) == 5200 and np.isfinite(dqn_losses[-1])
print({"final mean TD loss": round(float(np.mean(dqn_losses[-100:])), 4), "replay transitions": len(replay), "learned start-state torque": round(dqn_policy(np.array([1.0, 0.0, 0.0], dtype=np.float32)), 2)})
```

</details>

The implementation treats the teaching horizon as terminal. For an externally truncated continuing task, the target should usually bootstrap. Q-values can also be systematically overestimated because the same noisy estimates select and evaluate $\max_{a'}$; Double DQN separates those roles.


### **Policy Gradient Methods** {#policy-gradient-methods}

Value-based control chooses the action with the largest estimated value. Policy-gradient methods instead optimize a differentiable policy directly, which supports stochastic behavior and continuous actions. For $J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G(\tau)]$, the policy-gradient theorem gives

$$
\nabla_\theta J(\theta)=\mathbb{E}_{s,a\sim\pi_\theta}[\nabla_\theta\log\pi_\theta(a|s)Q^{\pi_\theta}(s,a)].
$$

The log-derivative converts an environment-dependent trajectory probability into gradients of policy probabilities. The environment dynamics need not be differentiable. Replacing $Q$ by $Q-b(s)$ does not change the expected gradient when the baseline is action-independent, but a good baseline can sharply reduce variance.

![Each sampled action receives a probability update weighted by its estimated advantage.](assets/dl17-policy-gradient.svg){fig-align="center" width="78%" fig-alt="A trajectory of states and actions ends in a return; log policy gradients are weighted by advantage."}

REINFORCE uses sampled return-to-go. It is simple and unbiased under the sampling assumptions, but one poor trajectory can dominate an update. Normalizing advantages changes finite-batch scaling; entropy bonuses slow premature collapse; gradient clipping limits extreme steps. None replaces multiple random seeds.

<details>
<summary><strong>PyTorch: train REINFORCE and inspect baseline centering</strong></summary>

```python
class CategoricalActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 48), nn.Tanh(), nn.Linear(48, 5))

    def forward(self, states):
        return torch.distributions.Categorical(logits=self.net(states))


def categorical_episode(actor, seed, horizon=120):
    env = LocalPendulum(horizon=horizon)
    state = env.reset(seed)
    states, actions, rewards = [], [], []
    previous_rng = torch.random.get_rng_state()
    torch.manual_seed(seed)
    for _ in range(horizon):
        state_tensor = torch.tensor(state, dtype=torch.float32)
        action_index = int(actor(state_tensor).sample())
        next_state, reward, _, truncated = env.step(float(discrete_torques[action_index]))
        states.append(state_tensor); actions.append(action_index); rewards.append(reward / 10.0)
        state = next_state
        if truncated:
            break
    torch.random.set_rng_state(previous_rng)
    return torch.stack(states), torch.tensor(actions), torch.tensor(rewards, dtype=torch.float32)


seed_everything(1740)
reinforce_actor = CategoricalActor()
reinforce_optimizer = torch.optim.Adam(reinforce_actor.parameters(), lr=2e-3)
reinforce_trace = []
for update in range(34):
    collected = [categorical_episode(reinforce_actor, 1740 + update * 7 + i) for i in range(4)]
    states = torch.cat([x[0] for x in collected])
    actions = torch.cat([x[1] for x in collected])
    returns_batch = torch.cat([discounted_returns(x[2]) for x in collected])
    advantages = (returns_batch - returns_batch.mean()) / (returns_batch.std() + 1e-6)
    distribution = reinforce_actor(states)
    loss = -(distribution.log_prob(actions) * advantages).mean() - 0.002 * distribution.entropy().mean()
    reinforce_optimizer.zero_grad(); loss.backward(); reinforce_optimizer.step()
    reinforce_trace.append(float(returns_batch[:120].mean()))

probe_states, probe_actions, probe_rewards = categorical_episode(reinforce_actor, 1799)
probe_returns = discounted_returns(probe_rewards)
centered_weights = probe_returns - probe_returns.mean()
assert abs(float(centered_weights.mean())) < 1e-5
print({"updates": len(reinforce_trace), "last return-to-go mean": round(reinforce_trace[-1], 3), "raw weight mean": round(float(probe_returns.mean()), 3), "centered weight mean": round(float(centered_weights.mean()), 6)})
```

</details>

A constant baseline centers the batch but does not explain state-dependent variation. A learned value baseline predicts which states naturally have larger future return, motivating actor-critic methods.


### **Actor-Critic Learning** {#actor-critic-learning}

An actor-critic agent maintains a policy (actor) and a value estimator (critic). The one-step TD residual

$$
\delta_t=r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)
$$

is simultaneously a critic error and a low-cost advantage estimate. The critic minimizes a regression loss; the actor increases $\log\pi_\theta(a_t|s_t)$ when $\delta_t>0$ and decreases it when $\delta_t<0$.

![The actor changes behavior while the critic turns reward and future value into an advantage signal.](assets/dl17-actor-critic.svg){fig-align="center" width="76%" fig-alt="State features branch into an actor and critic, whose outputs meet in a TD advantage."}

Sharing an encoder reduces computation but couples gradients from policy and value losses. A critic that learns too slowly gives noisy advantages; an overfit critic gives confidently biased ones. Common diagnostics include value loss, predicted-value scale, explained variance, policy entropy, gradient norms, and actor/critic learning-rate sensitivity.

<details>
<summary><strong>PyTorch: jointly update a categorical actor and state-value critic</strong></summary>

```python
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(3, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.actor = nn.Linear(64, 5)
        self.critic = nn.Linear(64, 1)

    def forward(self, states):
        features = self.trunk(states)
        return torch.distributions.Categorical(logits=self.actor(features)), self.critic(features).squeeze(-1)


class ActorView:
    def __call__(self, states):
        return a2c(states)[0]


seed_everything(1750)
a2c = ActorCritic()
a2c_optimizer = torch.optim.Adam(a2c.parameters(), lr=1.5e-3)
a2c_value_losses = []
for episode_id in range(90):
    states, actions, rewards = categorical_episode(ActorView(), 1750 + episode_id)
    returns_batch = discounted_returns(rewards)
    distribution, values = a2c(states)
    advantages = returns_batch - values.detach()
    actor_loss = -(distribution.log_prob(actions) * advantages).mean()
    critic_loss = F.smooth_l1_loss(values, returns_batch)
    loss = actor_loss + 0.5 * critic_loss - 0.003 * distribution.entropy().mean()
    a2c_optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(a2c.parameters(), 5.0)
    a2c_optimizer.step()
    a2c_value_losses.append(float(critic_loss.detach()))


def a2c_policy(observation):
    with torch.no_grad():
        distribution, _ = a2c(torch.tensor(observation, dtype=torch.float32))
    return float(discrete_torques[int(distribution.probs.argmax())])


assert np.isfinite(a2c_value_losses).all()
print({"last critic loss": round(a2c_value_losses[-1], 3), "median last-20 critic loss": round(float(np.median(a2c_value_losses[-20:])), 3), "upright torque": round(a2c_policy(np.array([1.0, 0.0, 0.0], dtype=np.float32)), 2)})
```

</details>

This compact implementation uses Monte Carlo returns for a stable teaching target even though the TD residual motivates actor-critic learning. Production A2C/A3C implementations normally collect fixed rollout fragments and bootstrap the critic at the fragment boundary.


### **Proximal Policy Optimization** {#proximal-policy-optimization}

Several epochs over one on-policy batch are sample-efficient, but after the first update the data came from an older policy. PPO measures $r_t(\theta)=\pi_\theta(a_t|s_t)/\pi_{\theta_{\mathrm{old}}}(a_t|s_t)$ and optimizes

$$
L^{\mathrm{clip}}(\theta)=\mathbb{E}_t[\min(r_t(\theta)\hat A_t,\operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\hat A_t)].
$$

![PPO removes the incentive for probability ratios to move beyond the clipping interval.](assets/dl17-ppo-clipping.svg){fig-align="center" width="76%" fig-alt="Piecewise surrogate curves show clipping around probability ratios one minus epsilon and one plus epsilon."}

For positive advantage, increasing the chosen action probability beyond $1+\epsilon$ gives no extra surrogate gain. For negative advantage, decreasing it below $1-\epsilon$ is similarly clipped. PPO does **not** guarantee a strict trust region: clipping only affects sampled objective terms. Approximate KL divergence, clip fraction, entropy, and value error must still be monitored.

<details>
<summary><strong>PyTorch: collect rollouts, compute GAE, and apply PPO clipping</strong></summary>

```python
class GaussianActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(3, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.mean = nn.Linear(64, 1)
        self.value = nn.Linear(64, 1)
        self.log_std = nn.Parameter(torch.tensor([-0.35]))

    def distribution_value(self, states):
        features = self.trunk(states)
        return torch.distributions.Normal(self.mean(features), self.log_std.exp()), self.value(features).squeeze(-1)

    @staticmethod
    def log_prob(distribution, raw_action):
        squashed = torch.tanh(raw_action)
        return (distribution.log_prob(raw_action) - torch.log(1 - squashed.square() + 1e-6)).sum(-1)


def collect_ppo_episode(model, seed):
    env, state = LocalPendulum(horizon=120), None
    state = env.reset(seed)
    states, raw_actions, log_probs, rewards, values = [], [], [], [], []
    torch.manual_seed(seed)
    for _ in range(120):
        state_tensor = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            distribution, value = model.distribution_value(state_tensor)
            raw_action = distribution.sample()
            log_prob = model.log_prob(distribution, raw_action)
        next_state, reward, _, truncated = env.step(float(2.0 * torch.tanh(raw_action)))
        states.append(state_tensor); raw_actions.append(raw_action); log_probs.append(log_prob)
        rewards.append(reward / 10.0); values.append(value)
        state = next_state
        if truncated:
            break
    rewards = torch.tensor(rewards, dtype=torch.float32)
    values = torch.stack(values)
    next_values = torch.cat([values[1:], torch.zeros(1)])
    advantages = generalized_advantages(rewards, values, next_values, gamma=0.98, lam=0.95)
    return torch.stack(states), torch.stack(raw_actions), torch.stack(log_probs), advantages, advantages + values


seed_everything(1760)
ppo = GaussianActorCritic()
ppo_optimizer = torch.optim.Adam(ppo.parameters(), lr=1.2e-3)
ppo_clip_fractions = []
for update in range(16):
    batches = [collect_ppo_episode(ppo, 1760 + update * 5 + offset) for offset in range(4)]
    states = torch.cat([x[0] for x in batches])
    raw_actions = torch.cat([x[1] for x in batches])
    old_log_probs = torch.cat([x[2] for x in batches])
    advantages = torch.cat([x[3] for x in batches])
    value_targets = torch.cat([x[4] for x in batches])
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-6)
    for _ in range(4):
        distribution, values = ppo.distribution_value(states)
        new_log_probs = ppo.log_prob(distribution, raw_actions)
        ratio = (new_log_probs - old_log_probs).exp()
        policy_loss = -torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages).mean()
        value_loss = F.smooth_l1_loss(values, value_targets)
        total_loss = policy_loss + 0.5 * value_loss - 0.002 * distribution.entropy().sum(-1).mean()
        ppo_optimizer.zero_grad(); total_loss.backward()
        nn.utils.clip_grad_norm_(ppo.parameters(), 1.0)
        ppo_optimizer.step()
    ppo_clip_fractions.append(float(((ratio - 1).abs() > 0.2).float().mean()))


def ppo_policy(observation):
    with torch.no_grad():
        distribution, _ = ppo.distribution_value(torch.tensor(observation, dtype=torch.float32))
    return float(2.0 * torch.tanh(distribution.mean))


assert 0 <= ppo_clip_fractions[-1] <= 1
print({"updates": 16, "last clip fraction": round(ppo_clip_fractions[-1], 3), "learned log standard deviation": round(float(ppo.log_std.detach()), 3)})
```

</details>

The code stores the pre-`tanh` action so that new and old log probabilities use the same change-of-variables correction. Omitting that Jacobian silently optimizes the wrong bounded-action density. Other common bugs include leaking gradients into old log probabilities, failing to normalize observations, and bootstrapping incorrectly at time limits.


### **Soft Actor-Critic** {#soft-actor-critic}

Soft Actor-Critic (SAC) is an off-policy actor-critic method for continuous actions. It maximizes reward and policy entropy:

$$
J(\pi)=\mathbb{E}_{\tau\sim\pi}\left[\sum_t\gamma^t(r_t+\alpha\mathcal{H}(\pi(\cdot|s_t)))\right].
$$

$\alpha$ is the temperature. Large values favor diverse actions; small values favor exploitation. The soft target is

$$
y=r+\gamma(1-d)\left[\min_iQ_{\bar\phi_i}(s',a')-\alpha\log\pi_\theta(a'|s')\right],\quad a'\sim\pi_\theta(\cdot|s').
$$

![SAC uses replay, twin critics, a squashed stochastic actor, and entropy regularization.](assets/dl17-sac-objective.svg){fig-align="center" width="76%" fig-alt="Replay data feed twin critics and a squashed Gaussian actor, which combine in a maximum-entropy objective."}

Twin critics use the smaller estimate to reduce optimistic error. The actor uses reparameterization $u=\mu_\theta(s)+\sigma_\theta(s)\epsilon$, $a=2\tanh u$, allowing gradients through sampled actions. Target critics are updated slowly. Automatic temperature tuning can match a target entropy instead of fixing $\alpha$.

<details>
<summary><strong>PyTorch: audit SAC critic and actor objectives on shared replay</strong></summary>

```python
class SquashedGaussianActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU())
        self.mean, self.log_std = nn.Linear(64, 1), nn.Linear(64, 1)

    def sample(self, states):
        features = self.body(states)
        mean = self.mean(features)
        log_std = self.log_std(features).clamp(-5, 1)
        distribution = torch.distributions.Normal(mean, log_std.exp())
        raw = distribution.rsample()
        squashed = torch.tanh(raw)
        action = 2.0 * squashed
        log_prob = distribution.log_prob(raw) - torch.log(2.0 * (1 - squashed.square()) + 1e-6)
        return action, log_prob.sum(-1, keepdim=True)


class ContinuousQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, states, actions):
        return self.net(torch.cat([states, actions], dim=-1))


seed_everything(1770)
sac_actor, sac_q1, sac_q2 = SquashedGaussianActor(), ContinuousQ(), ContinuousQ()
sac_t1, sac_t2 = copy.deepcopy(sac_q1), copy.deepcopy(sac_q2)
sac_actor_opt = torch.optim.Adam(sac_actor.parameters(), lr=8e-4)
sac_q_opt = torch.optim.Adam(list(sac_q1.parameters()) + list(sac_q2.parameters()), lr=8e-4)
sac_rng, alpha = np.random.default_rng(1770), 0.15

for update in range(650):
    ids = torch.tensor(sac_rng.integers(0, len(train_s), size=128))
    states, actions = train_s[ids], train_a[ids]
    rewards, next_states, dones = train_r[ids] / 10.0, train_ns[ids], train_done[ids]
    with torch.no_grad():
        next_actions, next_log_prob = sac_actor.sample(next_states)
        soft_next_q = torch.minimum(sac_t1(next_states, next_actions), sac_t2(next_states, next_actions))
        q_target = rewards + 0.98 * (1 - dones) * (soft_next_q - alpha * next_log_prob)
    q_loss = F.mse_loss(sac_q1(states, actions), q_target) + F.mse_loss(sac_q2(states, actions), q_target)
    sac_q_opt.zero_grad(); q_loss.backward(); sac_q_opt.step()

    sampled_actions, log_prob = sac_actor.sample(states)
    actor_loss = (alpha * log_prob - torch.minimum(sac_q1(states, sampled_actions), sac_q2(states, sampled_actions))).mean()
    sac_actor_opt.zero_grad(); actor_loss.backward(); sac_actor_opt.step()
    with torch.no_grad():
        for target_parameter, parameter in zip(sac_t1.parameters(), sac_q1.parameters()):
            target_parameter.mul_(0.995).add_(parameter, alpha=0.005)
        for target_parameter, parameter in zip(sac_t2.parameters(), sac_q2.parameters()):
            target_parameter.mul_(0.995).add_(parameter, alpha=0.005)


def sac_policy(observation):
    with torch.no_grad():
        features = sac_actor.body(torch.tensor(observation, dtype=torch.float32))
        return float(2.0 * torch.tanh(sac_actor.mean(features)))


assert torch.isfinite(q_loss) and torch.isfinite(actor_loss)
print({"critic loss": round(float(q_loss.detach()), 4), "actor loss": round(float(actor_loss.detach()), 4), "sample action range": (round(float(sampled_actions.min().detach()), 3), round(float(sampled_actions.max().detach()), 3))})
```

</details>

This audit reuses the chapter's **fixed logged dataset** and is therefore not a standard online SAC benchmark. Online SAC alternates interaction and replay updates so the buffer follows the improving policy. On a fixed dataset, actor actions can leave data support; the offline-RL section explains why an unconstrained SAC objective may exploit critic error.


### **Exploration and Credit Assignment** {#exploration-credit-assignment}

Exploration asks which experience to collect; credit assignment asks which past decisions caused later outcomes. They interact: an agent cannot assign credit to behavior it never tries, and sparse delayed rewards provide little guidance for deciding what to try next.

Action-space strategies include $\epsilon$-greedy exploration, policy entropy, parameter noise, optimistic uncertainty, and temporally correlated control noise. State-space strategies include counts, prediction error, information gain, and curricula. A novelty signal can fail when stochastic observations remain permanently unpredictable, the “noisy TV” problem.

An $n$-step target is $G_t^{(n)}=\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^nV(s_{t+n})$. GAE forms an exponentially weighted sum of TD residuals,

$$
\hat A_t^{\mathrm{GAE}(\gamma,\lambda)}=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}.
$$

<details>
<summary><strong>Python: inspect how GAE lambda changes temporal credit</strong></summary>

```python
credit_rewards = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])
credit_values = torch.tensor([0.15, 0.18, 0.22, 0.30, 0.40])
credit_next = torch.cat([credit_values[1:], torch.zeros(1)])
gae_by_lambda = {lam: generalized_advantages(credit_rewards, credit_values, credit_next, gamma=0.99, lam=lam) for lam in (0.0, 0.5, 0.95, 1.0)}
assert torch.allclose(gae_by_lambda[0.0], credit_rewards + 0.99 * credit_next - credit_values)
print({f"lambda={lam}": [round(float(x), 3) for x in advantages] for lam, advantages in gae_by_lambda.items()})
```

</details>

At $\lambda=0$, only the immediate TD residual receives credit. At $\lambda=1$, later residuals propagate through the full suffix. The best value depends on critic accuracy, horizon, reward delay, and batch size. Reward shaping must preserve the intended objective; an easy dense proxy can produce reward hacking rather than useful exploration.


### **Imitation and Offline Reinforcement Learning** {#imitation-offline-reinforcement-learning}

Behavioral cloning (BC) minimizes $\mathcal{L}_{\mathrm{BC}}=-\mathbb{E}_{(s,a)\sim\mathcal D}\log\pi_\theta(a|s)$. Deployment errors can move the policy into states absent from demonstrations, where further errors compound. DAgger queries an expert on learner-visited states, but that option is unavailable when collection is fixed.

Offline RL tries to improve a policy using only static data. Standard Bellman backups evaluate actions proposed by the learned policy, including actions poorly represented in $\mathcal D$. Function approximation can assign those actions spuriously large values.

![An offline critic may mistake extrapolation error outside the logged support for a high-value action.](assets/dl17-offline-shift.svg){fig-align="center" width="76%" fig-alt="Logged state-action points occupy one region while a learned policy moves toward an unsupported action outside that region."}

Conservative Q-learning adds a penalty. In a discrete form,

$$
\mathcal R_{\mathrm{CQL}}=\mathbb E_{s\sim\mathcal D}\left[\log\sum_a\exp Q(s,a)-Q(s,a_{\mathcal D})\right].
$$

The [CQL paper](https://proceedings.neurips.cc/paper/2020/hash/0d2b2061826a5df3221116a5085a6052-Abstract.html) studies conservative value estimation under distribution shift. Excessive conservatism can prevent improvement beyond the behavior policy.

<details>
<summary><strong>PyTorch: behavioral cloning and a continuous-action CQL penalty</strong></summary>

```python
class DeterministicActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, states):
        return 2.0 * torch.tanh(self.net(states))


seed_everything(1780)
bc_actor = DeterministicActor()
bc_optimizer = torch.optim.AdamW(bc_actor.parameters(), lr=1e-3, weight_decay=1e-4)
for _ in range(500):
    ids = torch.randint(len(train_s), (128,))
    bc_loss = F.mse_loss(bc_actor(train_s[ids]), train_a[ids])
    bc_optimizer.zero_grad(); bc_loss.backward(); bc_optimizer.step()

with torch.no_grad():
    validation_bc_mse = F.mse_loss(bc_actor(val_s), val_a)
    sampled_random_actions = torch.empty(len(val_s), 12, 1).uniform_(-2, 2)
    repeated_states = val_s[:, None, :].expand(-1, 12, -1)
    random_q = sac_q1(repeated_states.reshape(-1, 3), sampled_random_actions.reshape(-1, 1)).reshape(len(val_s), 12)
    data_q = sac_q1(val_s, val_a).squeeze(1)
    cql_penalty = torch.logsumexp(random_q, dim=1).mean() - data_q.mean()


def bc_policy(observation):
    with torch.no_grad():
        return float(bc_actor(torch.tensor(observation, dtype=torch.float32)))


assert validation_bc_mse >= 0 and torch.isfinite(cql_penalty)
print({"BC validation action MSE": round(float(validation_bc_mse), 4), "CQL support penalty": round(float(cql_penalty), 4), "logged action std": round(float(train_a.std()), 3)})
```

</details>

The random-action log-sum-exp is a Monte Carlo approximation for continuous actions, not a complete CQL implementation. A reliable offline study characterizes coverage, behavior quality, episode boundaries, reward relabeling, and whether model selection used online interaction. Test-environment access during tuning can quietly invalidate the offline claim.


### **Model-Based Reinforcement Learning** {#model-based-reinforcement-learning}

Model-based RL learns or uses $\hat p_\psi(s_{t+1},r_t|s_t,a_t)$ and then plans, generates synthetic experience, or differentiates through the model. A model can reuse each real transition many times. Its central cost is **model bias**: a policy may visit states where learned dynamics are wrong and exploit those errors.

![One-step prediction error can accumulate into a large trajectory error.](assets/dl17-model-rollout-bias.svg){fig-align="center" width="76%" fig-alt="Real and learned trajectories begin together and then diverge as rollout horizon increases."}

An ensemble approximates epistemic uncertainty through different initialization or resampling. Planning can penalize disagreement. Short model rollouts branched from replay, as studied in [MBPO](https://proceedings.neurips.cc/paper/2019/hash/5faf461eff3099671ad63c6f3f094f7f-Abstract.html), limit compounding error while adding synthetic transitions.

<details>
<summary><strong>PyTorch: fit a dynamics ensemble and measure rollout error by horizon</strong></summary>

```python
class DynamicsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 96), nn.SiLU(), nn.Linear(96, 96), nn.SiLU(), nn.Linear(96, 4))


input_mean = torch.cat([train_s, train_a], dim=1).mean(0)
input_std = torch.cat([train_s, train_a], dim=1).std(0).clamp_min(1e-4)
target_train = torch.cat([train_ns - train_s, train_r / 10.0], dim=1)
target_mean, target_std = target_train.mean(0), target_train.std(0).clamp_min(1e-4)


def normalized_dynamics(model, states, actions):
    normalized_input = (torch.cat([states, actions], dim=1) - input_mean) / input_std
    prediction = model.net(normalized_input) * target_std + target_mean
    return states + prediction[:, :3], prediction[:, 3:4]


ensemble = []
for model_id in range(3):
    seed_everything(1790 + model_id)
    model = DynamicsModel()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(1790 + model_id)
    for _ in range(550):
        ids = torch.randint(len(train_s), (160,), generator=generator)
        normalized_input = (torch.cat([train_s[ids], train_a[ids]], 1) - input_mean) / input_std
        normalized_target = (target_train[ids] - target_mean) / target_std
        loss = F.mse_loss(model.net(normalized_input), normalized_target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    ensemble.append(model.eval())

with torch.no_grad():
    next_predictions = torch.stack([normalized_dynamics(model, val_s, val_a)[0] for model in ensemble])
    one_step_mse = F.mse_loss(next_predictions.mean(0), val_ns)
    disagreement = next_predictions.var(0).mean()

heldout = test_episodes[0]
predicted = torch.tensor(heldout[0][0], dtype=torch.float32).unsqueeze(0)
rollout_errors = []
with torch.no_grad():
    for transition in heldout[:40]:
        action = torch.tensor([[transition[1]]], dtype=torch.float32)
        candidates = torch.stack([normalized_dynamics(model, predicted, action)[0] for model in ensemble])
        predicted = candidates.mean(0)
        true_next = torch.tensor(transition[3], dtype=torch.float32).unsqueeze(0)
        rollout_errors.append(float(F.mse_loss(predicted, true_next)))

assert rollout_errors[-1] >= 0 and torch.isfinite(one_step_mse)
print({"validation one-step MSE": round(float(one_step_mse), 5), "ensemble disagreement": round(float(disagreement), 6), "open-loop MSE": {"h=1": round(rollout_errors[0], 5), "h=10": round(rollout_errors[9], 5), "h=40": round(rollout_errors[39], 5)}})
```

</details>

One-step validation error is necessary but not sufficient: planning changes the state-action distribution, and repeated predictions feed model output back as input. Useful diagnostics include horizon-conditioned error, ensemble disagreement, calibration, constraint violations, and real-versus-model return gaps under the candidate policy.


### **World Models** {#world-models}

A world model learns a compact predictive state used for planning or policy learning. The original [World Models](https://worldmodels.github.io/) system separated vision $V$, recurrent memory $M$, and controller $C$. Modern latent-imagination agents such as [DreamerV3](https://arxiv.org/abs/2301.04104) jointly learn representations, latent dynamics, reward and continuation predictors, then train actor-critic components on imagined trajectories.

![The original World Models architecture separates vision, memory, and controller components.](assets/dl17-world-model-overview.svg){fig-align="center" width="74%" fig-alt="World Models overview with a vision encoder, recurrent memory model, controller, and environment."}

*Source: Ha and Schmidhuber, [World Models](https://worldmodels.github.io/), diagram licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) by the authors.*

A recurrent state-space model (RSSM) commonly combines deterministic memory $h_t$ and stochastic state $z_t$:

$$
h_t=f_\psi(h_{t-1},z_{t-1},a_{t-1}),\qquad p_\psi(z_t|h_t),\qquad q_\psi(z_t|h_t,o_t).
$$

The posterior $q$ incorporates the real observation during training; the prior $p$ predicts without it during imagination. Decoders predict observations or task-relevant features, reward, and continuation. A KL term aligns posterior and prior. Actor gradients through imagined trajectories can be efficient, but the actor may exploit model defects.

<details>
<summary><strong>PyTorch: train a compact recurrent world model and test open-loop imagination</strong></summary>

```python
def episode_windows(selected, length=20):
    states, actions, next_states, rewards = [], [], [], []
    for episode in selected:
        for start in range(0, len(episode) - length + 1, length):
            window = episode[start:start + length]
            states.append(np.stack([x[0] for x in window]))
            actions.append(np.array([[x[1]] for x in window], dtype=np.float32))
            next_states.append(np.stack([x[3] for x in window]))
            rewards.append(np.array([[x[2] / 10.0] for x in window], dtype=np.float32))
    return tuple(torch.tensor(np.stack(x), dtype=torch.float32) for x in (states, actions, next_states, rewards))


wm_train_s, wm_train_a, wm_train_ns, wm_train_r = episode_windows(train_episodes)
wm_val_s, wm_val_a, wm_val_ns, wm_val_r = episode_windows(val_episodes)


class TinyRecurrentWorldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(3, 24), nn.SiLU())
        self.recurrent = nn.GRU(input_size=25, hidden_size=48, batch_first=True)
        self.state_head = nn.Linear(48, 3)
        self.reward_head = nn.Linear(48, 1)

    def forward(self, states, actions, hidden=None):
        latent = self.encoder(states)
        hidden_sequence, final_hidden = self.recurrent(torch.cat([latent, actions], dim=-1), hidden)
        return self.state_head(hidden_sequence), self.reward_head(hidden_sequence), final_hidden


seed_everything(1800)
world_model = TinyRecurrentWorldModel()
world_optimizer = torch.optim.AdamW(world_model.parameters(), lr=1.5e-3, weight_decay=1e-4)
generator = torch.Generator().manual_seed(1800)
for _ in range(700):
    ids = torch.randint(len(wm_train_s), (32,), generator=generator)
    predicted_states, predicted_rewards, _ = world_model(wm_train_s[ids], wm_train_a[ids])
    world_loss = F.mse_loss(predicted_states, wm_train_ns[ids]) + 0.25 * F.mse_loss(predicted_rewards, wm_train_r[ids])
    world_optimizer.zero_grad(); world_loss.backward(); world_optimizer.step()

with torch.no_grad():
    val_state_prediction, _, _ = world_model(wm_val_s, wm_val_a)
    teacher_forced_mse = F.mse_loss(val_state_prediction, wm_val_ns)
    imagined_state = wm_val_s[:, 0]
    hidden = None
    imagination_errors = []
    for time_index in range(wm_val_a.shape[1]):
        state_prediction, _, hidden = world_model(imagined_state[:, None, :], wm_val_a[:, time_index:time_index + 1], hidden)
        imagined_state = state_prediction[:, 0]
        imagination_errors.append(float(F.mse_loss(imagined_state, wm_val_ns[:, time_index])))

assert len(imagination_errors) == 20 and torch.isfinite(teacher_forced_mse)
print({"teacher-forced state MSE": round(float(teacher_forced_mse), 5), "imagination MSE": {"h=1": round(imagination_errors[0], 5), "h=10": round(imagination_errors[9], 5), "h=20": round(imagination_errors[19], 5)}})
```

</details>

This deterministic GRU is a **mechanism model**, not a Dreamer reproduction: Pendulum is fully observed and low-dimensional, there is no stochastic posterior, image decoder, continuation model, or imagined actor update. Its purpose is to expose the gap between teacher-forced prediction and autonomous imagination. Low reconstruction error does not imply that latent states preserve control-relevant information.


### **Evaluation, Reproducibility, and Safety** {#evaluation-reproducibility-safety}

RL evaluation is unusually sensitive to randomness because data collection, exploration, initialization, replay order, and environment starts interact. A single training seed or best checkpoint is not evidence of robustness. [Deep Reinforcement Learning that Matters](https://arxiv.org/abs/1709.06560) documents how implementation and reporting choices can alter conclusions.

![A defensible RL result separates training seeds, fixed evaluation starts, metrics, and uncertainty reporting.](assets/dl17-evaluation-protocol.svg){fig-align="center" width="78%" fig-alt="A pipeline shows multiple training seeds evaluated on fixed starts and reported with returns, safety metrics, and uncertainty intervals."}

A defensible protocol records multiple training seeds, fixed test starts, deterministic and stochastic evaluation, curves against environment steps and wall time, return distributions, termination conventions, wrappers, and safety violations. Reward is an engineered proxy: an agent can maximize it through specification gaming, unsafe exploration, simulator artifacts, or model exploitation.

<details>
<summary><strong>Python: compare policies on fixed starts with bootstrap intervals and safety diagnostics</strong></summary>

```python
def evaluate_policy(policy, seeds, horizon=120):
    returns, saturation_rates, peak_speeds = [], [], []
    for seed in seeds:
        env = LocalPendulum(horizon=horizon)
        state = env.reset(int(seed))
        episode_return, actions, speeds = 0.0, [], []
        for _ in range(horizon):
            action = float(np.clip(policy(state), -2, 2))
            state, reward, _, truncated = env.step(action)
            episode_return += reward; actions.append(action); speeds.append(abs(float(state[2])))
            if truncated:
                break
        returns.append(episode_return)
        saturation_rates.append(np.mean(np.abs(actions) > 1.95))
        peak_speeds.append(max(speeds))
    return np.array(returns), float(np.mean(saturation_rates)), float(np.mean(peak_speeds))


def bootstrap_mean_interval(values, seed=1810, draws=2000):
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(draws, len(values)), replace=True).mean(1)
    return tuple(np.quantile(samples, [0.025, 0.975]))


evaluation_seeds = np.arange(18_100, 18_112)
policies = {"zero torque": lambda state: 0.0, "DQN": dqn_policy, "A2C": a2c_policy, "PPO": ppo_policy, "offline BC": bc_policy, "offline SAC objective": sac_policy}
evaluation = {}
for name, policy in policies.items():
    returns_array, saturation, peak_speed = evaluate_policy(policy, evaluation_seeds)
    low, high = bootstrap_mean_interval(returns_array, seed=1810 + len(name))
    evaluation[name] = {"mean return": round(float(returns_array.mean()), 1), "95% bootstrap interval": (round(float(low), 1), round(float(high), 1)), "torque saturation": round(saturation, 3), "mean peak speed": round(peak_speed, 2)}

assert all(np.isfinite(item["mean return"]) for item in evaluation.values())
assert all(item["95% bootstrap interval"][0] <= item["95% bootstrap interval"][1] for item in evaluation.values())
print(evaluation)
```

</details>

The interval quantifies variation across fixed start states, not training-seed uncertainty: each policy was trained once. The numbers compare notebook mechanisms under one small budget; they do not rank DQN, PPO, SAC, or BC in general. A benchmark claim requires independent training seeds, tuned baselines, confidence intervals across runs, and an established implementation.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Deep RL methods differ by **what they learn**, **where data comes from**, and **whether they use an environment model**.

| Method | Learned object | Data regime | Action space | Main strength | Characteristic risk |
|---|---|---|---|---|---|
| DQN | action-value function | online, off-policy replay | discrete | reuses experience | bootstrap instability and overestimation |
| REINFORCE | stochastic policy | online, on-policy | either | direct and simple | high gradient variance |
| Actor-Critic | policy and critic | usually online | either | lower-variance updates | biased or lagging critic |
| PPO | clipped actor-critic objective | online, on-policy | either | robust practical baseline | costly fresh rollouts; clipping is no guarantee |
| SAC | entropy actor and twin critics | online, off-policy replay | continuous | replay reuse and exploration | critic support and implementation sensitivity |
| Behavioral cloning | policy | fixed demonstrations | either | stable supervised objective | compounding covariate shift |
| Offline RL / CQL | conservative policy and value | fixed logged data | either | no new interaction | dataset support and selection leakage |
| Model-based RL | dynamics/reward plus planner or policy | online or offline | either | sample-efficient simulation | compounding model bias |
| World-model agent | latent dynamics, reward, actor, critic | real and imagined trajectories | either | policy learning in imagination | exploitation of latent-model errors |

A practical sequence is:

1. Define state, action, reward, horizon, termination, constraints, and evaluation before choosing an algorithm.
2. Use DQN for meaningful finite actions; PPO as a clear on-policy baseline; SAC when continuous control and replay efficiency matter.
3. Use imitation when demonstrations exist, but audit deployment covariate shift.
4. Treat static logged data as offline RL and measure action support explicitly.
5. Add learned dynamics when interaction cost justifies managing model bias.
6. Use latent world models with open-loop prediction, uncertainty, and real-environment validation.
7. Report distributions across seeds and starts, learning cost, safety metrics, and failures alongside mean return.

This chapter stops before using RL for preference alignment. PPO reappears next as one component of RLHF, where prompts, generated responses, reward models, KL constraints, and human-preference data change both the objective and its failure modes.
